In [1]:
import numpy as np

class asset_pricing():      # A class can work as a set of function that I can call at once. 
    '''This is the asset pricing model from the course'''
    def __init__(self, beta=0.99,c = 10):      # we must always in __init__ functions start with "self", 
        '''Initializer, '''                     # because when python calls objects the object itself is always the first information.
                                                # So here we define a function with parameters. 
        self.beta = beta 
        self.c = c                          # self. is to be written before every object inside a class

    def belmann(self, V0):
        '''This updates the V, so it takes the value of the iteration'''
        V1 = self.c+self.beta*V0
        return V1
    
    def solve(self, V0=0.0, maxiter = 1000, tol = 1e-14):   # sætter startværdi, max iterationer og toleranceniveau. Jeg kalder bare funktionen for solve jo... omg
        ''' solves the model using value function iterations'''
        V = np.array([V0])      # Vi starter med at lave en array med V0. Den kan vi altid tilføje ting til 
        for iter in range(maxiter): 
            V1 = self.belmann(V[iter])      # vi definerer V1, altid, som belmann funktion til V_i
                                            # når vi skriver V[iter], så tager vi 'iter' værdien af arrayen V. 
            V = np.append(V,V1)             # Vi appender V med V (som jo er hele arrayen indtil nu) og bagefter med V1 som er den seneste iteration. 

            if abs(V[iter+1]-V[iter]) < tol: 
                print('Convergence achieved after ', iter, 'iterations')
                print('Numerical solutions', round(V[iter+1], 16))
                print('Convergence achieved after', iter, 'iterations')
                return V
            
        else: 
            print('No convergence after', maxiter, 'iterations')
            return V
model = asset_pricing(beta = 0.995, c = 100)             # her der giver vi modellen nogle værdier

V = model.solve(maxiter=10000,tol=1e-14)                # her der kalder vi på modellen. 


Convergence achieved after  6289 iterations
Numerical solutions 19999.99999999962
Convergence achieved after 6289 iterations


Okay, but would this be possible to do with our code where utility is log of consumption and where consumption is given as the amount of assets taken and wages? 

No. What we are working with here is actually a case of the cake eating problem. Because every future period your value depends on the amount of cake left. Except that here there is constantly added cake. 

In [45]:
grider = np.linspace(0,10,11)
c = (grider - grider[:, np.newaxis])
print(c)

[[  0.   1.   2.   3.   4.   5.   6.   7.   8.   9.  10.]
 [ -1.   0.   1.   2.   3.   4.   5.   6.   7.   8.   9.]
 [ -2.  -1.   0.   1.   2.   3.   4.   5.   6.   7.   8.]
 [ -3.  -2.  -1.   0.   1.   2.   3.   4.   5.   6.   7.]
 [ -4.  -3.  -2.  -1.   0.   1.   2.   3.   4.   5.   6.]
 [ -5.  -4.  -3.  -2.  -1.   0.   1.   2.   3.   4.   5.]
 [ -6.  -5.  -4.  -3.  -2.  -1.   0.   1.   2.   3.   4.]
 [ -7.  -6.  -5.  -4.  -3.  -2.  -1.   0.   1.   2.   3.]
 [ -8.  -7.  -6.  -5.  -4.  -3.  -2.  -1.   0.   1.   2.]
 [ -9.  -8.  -7.  -6.  -5.  -4.  -3.  -2.  -1.   0.   1.]
 [-10.  -9.  -8.  -7.  -6.  -5.  -4.  -3.  -2.  -1.   0.]]


So the below code creates the bellman function for the cake eating problem with grid = 50
But we have to loop through the solutions to solve it correctly. 

In [28]:
# Here I copy the code from the cake eating problem class

class cake_ongrid(): 
    
    def __init__(self, beta = 0.99, wbar = 10, ngrid = 50): 
        self.beta = beta        # discount factor
        self.wbar = wbar        # max cake size
        self.ngrid = ngrid      # grid size 
        self.epsilon = np.finfo(float).eps     # I don't know what is happening, but this is a way to get the smallest number possible in python. 
        self.grid = np.linspace(self.epsilon, self.wbar, self.ngrid)    # np.linspace(start, stop, number of points), so an array from 0 to wbar with 50 points. 
                                                                        
    
    def bellman(self, V0):
        """Bellman operator, V0 is one-dim vector of values on grid"""
        c = (self.grid-self.grid[:, np.newaxis] )                       # this is c = W - W_next. W is in the column and chocie in rows. 
                                                                        # look at top row in each column for state. Then for each state you can take some decicions that leave you with wnext
                                                                        # So column 5 is w = 5. Row 4 = consumption = 4. So, Column 5 row four means we have w = 5 and use for so we are left with 1 for wnext. 
        c[c==0] = self.epsilon    # So log 0 will be defined. 
        mask = c > 0    # So now we are only looking at the choices in c that are actually possible. 
        matV1 = np.full((self.ngrid, self.ngrid), -np.inf)          # We just do this so we are sure that the 'mask' thing works. 
        matV0 = np.repeat(V0[:, None], self.ngrid, axis=1)    # Here I change the 1-dim vector into an array of [n,1], n rows and one column, that is V0[:,None]
                                                              # Then I repeat it ngrid times along the columns, to get a [n,ngrid] array of the current value function. 
        matV1[mask] = np.log(c[mask]) + self.beta*matV0[mask]    # matV1[mask] giver en flad 1D array, kun med de positive = mulige værdier.
                                                                # så her får vi summen af alle de mulige værdier af log(c) + beta*V0.
        V1 = np.max(matV1, axis = 0)
        c1 = [self.grid - self.grid[np.argmax(matV1, axis = 0)]]    # taken the array of the optimal value in matV1
        return V1, c1

model = cake_ongrid(beta=0.95, wbar=10, ngrid=50)       # så det vi definerer her er 'model', som tager det output, der kommer af class ' cake_ongrid(): med de inputværdier som vi har valgt her. 
V0 = np.log(model.grid)
V1, c1 = model.bellman(V0)                                 
                                                                                                                                   

Det vi skal have er jo værdien af value function, og så for alle initialværdier af A, så vi kan se, hvad værdien er der, og bruge det til backwards induction. 

In [17]:
from time import process_time

In [30]:
def vfi_solve(self, maxiter = 1000, tol = 1e-4, callback = None): 
    """Solves the model using VFI (successive approximations)"""
    tic = process_time()    # Why do we need to start a time counter? 

    V0 = np.log(self.grid)      # så vi definerer V0 til at være en array, hvor vi bare spiser hele kagen (altså, alle mulige værdier af kagen og tager nytten af dem)
    for iter in range(maxiter): 
        V1, c1 = self.bellman(V0)      # Så vi kalder bellman funktionen, som spytter V1 og C1 ud. Se ovenfor. 
        
        if np.all(abs(V1 - V0) < tol):
            toc = process_time()
            print("Convergence achieved after", iter,"iterations and", round(toc-tic, 5), "seconds")
        else: 
            print("No convergence achieved after ", maxiter, "iterations")
            break 
        V0 = V1     # here we take the new array that we've made with the bellman equation function and adds that to V0, so we don't have to change the name to Vn for every iteration. 
        print(V1.shape)
    return V1, c1


cake_ongrid.solve = vfi_solve           # So now this function is inside the 'cake_ongrid' class and it is turned into a method. 

In [35]:
V1 = model.solve(maxiter=1000, tol=1e-4)



No convergence achieved after  1000 iterations
